# Data Preprocessing and Exploration

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Importing Datasets

Original 'dirty' datasets retrieved from AWS s3 bucket and aed dataset with geocoded addresses retrieved from public GitHub Repository.

In [ ]:
# Setting parameters for Papermill
url = 's3://mdaprojectdata2/'
url2 = 'https://raw.githubusercontent.com/JeroenGuillierme/Project-MDA/main/Data/'
output_data_path = 'Results/preprocessed_data.csv'

In [ ]:
ambulance = pd.read_parquet(f'{url}ambulance_locations.parquet.gzip')
mug = pd.read_parquet(f'{url}mug_locations.parquet.gzip')
pit = pd.read_parquet(f'{url}pit_locations.parquet.gzip')
interventions1 = pd.read_parquet(f'{url}interventions1.parquet.gzip')
interventions2 = pd.read_parquet(f'{url}interventions2.parquet.gzip')
interventions3 = pd.read_parquet(f'{url}interventions3.parquet.gzip')
interventions4 = pd.read_parquet(f'{url}interventions_bxl.parquet.gzip')
interventions5 = pd.read_parquet(f'{url}interventions_bxl2.parquet.gzip')
cad = pd.read_parquet(f'{url}cad9.parquet.gzip')
aed = pd.read_parquet(f'{url}aed_locations.parquet.gzip')

aed_total = pd.read_csv(f'{url2}aed_total_coordinates.csv')
mug1 = pd.read_csv(f'{url2}mug1.csv')

# Load Belgium with regions shapefile
belgium_with_provinces_boundary = gpd.read_file(f'{url2}BELGIUM_-_Provinces.geojson')

# External PIT data with their latitudes and longitudes
pit_locations = pd.read_excel(f'{url2}pit_01022024_fr.xlsx')

pd.set_option('display.max_columns', None)
pd.options.mode.copy_on_write = True

In [ ]:
# Ensure the CRS (Coordinate Reference System) is set to WGS84 (latitude/longitude)
belgium_with_provinces_boundary = belgium_with_provinces_boundary.to_crs(4326)

## 2. Functions

Here some custom functions are imported from the Functions.py script, which will be used further in this Notebook for preprocessing/cleaning the data.

In [ ]:
from Functions import (correct_latitude, correct_longitude, is_within_belgium, assign_province, assign_nearest_province, 
convert_format1, convert_format2, convert_format3, extract_numeric, timedelta_to_minutes, draw_histograms)

## 3. Preprocessing Interventions Datasets

### 3.1 Intervention datasets one till three

1) Only cardiac arrest incidents are kept in the datasets.
2) Time variables converted to datetime.
3) Conctenate first three intervention datasets together.
4) Create new variable for response time: T3-T0.
5) Correct format of the latitude and longitude coordinates.
6) Rename variables Eventlevel and EventType.
7) Select only relevant variables to go further with.

In [ ]:
# only keeping cardiac arrest interventions
interventions1 = interventions1[interventions1["EventType Firstcall"] == "P003 - Cardiac arrest"]
interventions2 = interventions2[interventions2["EventType Firstcall"] == "P003 - Cardiac arrest"]
interventions3 = interventions3[interventions3["EventType Firstcall"] == "P003 - Cardiac arrest"]

In [ ]:
# Converting time formats to datetime 
# T0
interventions1["T0"] = interventions1["T0"].apply(convert_format1)
interventions2["T0"] = interventions2["T0"].apply(convert_format1)
interventions3["T0"] = interventions3["T0"].apply(convert_format1)

# T3
interventions1["T3"] = interventions1["T3"].apply(convert_format2)
interventions2["T3"] = interventions2["T3"].apply(convert_format2)
interventions3["T3"] = interventions3["T3"].apply(convert_format2)


In [ ]:
# Concatenating first three intervention datasets
interventions123_total = pd.concat([interventions1, interventions2, interventions3], axis=0)

# Creating response time variable T3-T0, after converting them in the right date-time format
interventions123_total["T3-T0"] = interventions123_total["T3"] - interventions123_total["T0"]

# Creating new variable Intervention (binary indicator)
interventions123_total['Intervention'] = 1

# Correcting coordinates
interventions123_total['Latitude'] = interventions123_total['Latitude intervention'].apply(correct_latitude).copy()
interventions123_total['Longitude'] = interventions123_total['Longitude intervention'].apply(correct_longitude).copy()

# Renaming variable Eventlevel
interventions123_total['Eventlevel'] = interventions123_total["EventLevel Firstcall"].copy()


# Renaming variable EventType
interventions123_total['EventType'] = interventions123_total['EventType Firstcall']

# Only keep several variables
interventions123_total = interventions123_total[
    ['Mission ID', 'Latitude', 'Longitude', "Intervention", "Eventlevel", "T3-T0", "EventType", "Vector type", "Province intervention"]]

In [ ]:
interventions123_total.head(5)

### 3.2 cad dataset

1. Only cardiac arrest are kept in the cad dataset.
2. Time variable converted to datetime.
3. Creating new variables: Intervention (Binary indicator) and T3-T0 (response time).
4. Renaming variables Eventlevel, Vector type and Province intervention.
5. Correct format of the latitude and longitude coordinates.
6. Select only relevant variables to go further with.

In [ ]:
# Only keep cardiac arrest incidents
cad = cad[cad['EventType Trip'] == "P003 - HARTSTILSTAND - DOOD - OVERLEDEN"]

In [ ]:
# Converting time formats to datetime 
# T0
cad["T0"] = cad["T0"].apply(convert_format2)
# T3
cad["T3"] = cad["T3"].apply(convert_format2)

In [ ]:
# Create extra column Intervention
cad['Intervention'] = 1 

# Creating new variable response time T3-T0, after converting them in the right date-time format
cad['T3-T0'] = cad['T3'] - cad['T0']

# Renaming variable EventType
cad['EventType'] = cad['EventType Trip']

# Renaming variable EventLevel
cad['Eventlevel'] = cad['EventLevel Trip']

# Renaming variable Vector type
cad['Vector type'] = cad['Vector Type']

# Renaming variable Province intervention
cad['Province intervention'] = cad['Province invervention']

# Correcting coordinates
cad['Latitude'] = cad['Latitude intervention'].apply(correct_latitude)
cad['Longitude'] = cad['Longitude intervention'].apply(correct_longitude)

# Only keep several variables
cad = cad[['Mission ID', 'Latitude', 'Longitude', 'Intervention', 'Eventlevel', 'T3-T0', 'EventType', 'Vector type', 'Province intervention']]

In [ ]:
cad.head(5)

### 3.3 Intervention datasets from Brussels

1. Only cardiac arrest are kept in the datasets.
2. Time variables converted to datetime.
3. Creating new variables: Intervention (Binary indicator), T3-T0 (response time) and Province intervention.
4. Renaming variables EventType, Eventlevel, Vector type and Mission ID.
5. Correct format of the latitude and longitude coordinates.
6. Select only relevant variables to go further with.

In [ ]:
# Only keeping interventions for cardiac arrests
interventions4 = interventions4[interventions4['eventtype_firstcall'] == 'P003 - Cardiac arrest']
interventions5 = interventions5[
    (interventions5['EventType and EventLevel'] == 'P003  N01 - HARTSTILSTAND - DOOD - OVERLEDEN') | (
            interventions5['EventType and EventLevel'] == 'P003  N05 - HARTSTILSTAND - DOOD - OVERLEDEN')]

In [ ]:
# Converting time formats to datetime 
# T0
interventions4["t0"] = interventions4["t0"].apply(convert_format3)
interventions5["T0"] = interventions5["T0"].apply(convert_format1)
# T3
interventions4["t3"] = interventions4["t3"].apply(convert_format3)
interventions5["T3"] = interventions5["T3"].apply(convert_format1)

In [ ]:
# Creating variable Intervention
interventions4['Intervention'] = 1
interventions5['Intervention'] = 1

# Creating variable Response Time T3-T0
interventions4['T3-T0'] = interventions4['t3'] - interventions4['t0']
interventions5['T3-T0'] = interventions5['T3'] - interventions5['T0']

# Creating variable 'Province intervention'
interventions4['Province intervention'] = 'BXL'
interventions5['Province intervention'] = 'BXL'

# renaming variable EventType
interventions4['EventType'] = interventions4['eventtype_firstcall']
interventions5['EventType'] = interventions5['EventType and EventLevel'].apply(lambda x: re.sub(r'  [A-Z]\d{2}', '', x))

# Renaming variable Eventlevel
interventions4['Eventlevel'] = interventions4['eventLevel_firstcall']
interventions5["Eventlevel"] = interventions5["EventType and EventLevel"].str.split(" ").str[2].str.replace("0", "")

# Renaming variable Vector type
interventions4['Vector type'] = interventions4['vector_type']
interventions5['Vector type'] = interventions5['Vector type NL']

# Renaming variable Mission ID
interventions4['Mission ID'] = interventions4['mission_id']

# Correcting coordinates
interventions4['Latitude'] = interventions4['latitude_intervention'].apply(correct_latitude)
interventions4['Longitude'] = interventions4['longitude_intervention'].apply(correct_longitude)
interventions5['Latitude'] = interventions5['Latitude intervention'].apply(correct_latitude)
interventions5['Longitude'] = interventions5['Longitude intervention'].apply(correct_longitude)

# Only keeping several variables
interventions4 = interventions4[['Mission ID', 'Latitude', 'Longitude', 'Intervention', 'Eventlevel', 'T3-T0', 'EventType', 'Vector type', 'Province intervention']]
interventions5 = interventions5[['Mission ID', 'Latitude', 'Longitude', 'Intervention', 'Eventlevel', 'T3-T0', 'EventType', 'Vector type', 'Province intervention']]
                                 

In [ ]:
interventions4.head(5)

In [ ]:
interventions5.head(5)

### 3.4 Concatenating all intervention datasets together

1. Concatenate all interventions dataset together.
2. Create new variables: AED, Ambulance, Mug and PIT (Binary inicators)
3. Extract numeric part from Eventlevel variable.
4. Convert the T3-T0 variable from timedelta to minutes.
5. Drop observations with missing values for the variables Latitude or Longitude.
6. Reset the indices of the dataset.
7. Make the different notations for each level of the variable Vector type consistent.

In [ ]:
interventions_TOTAL = pd.concat([interventions123_total, cad, interventions4, interventions5], axis=0)

# Creating new AED column
interventions_TOTAL['AED'] = 0  

# Creating new columns for distance calculations later on
interventions_TOTAL['Ambulance'] = 0
interventions_TOTAL['Mug'] = 0
interventions_TOTAL['PIT'] = 0

# Extracting numeric part from variable Eventlevel
interventions_TOTAL['Eventlevel'] = interventions_TOTAL['Eventlevel'].apply(extract_numeric)

# Converting timedelta to minutes
interventions_TOTAL['T3-T0'] = interventions_TOTAL['T3-T0'].apply(timedelta_to_minutes)

# Drop rows with missing latitude or longitude
interventions_TOTAL = interventions_TOTAL.dropna(subset=['Latitude', 'Longitude'])

# Reset Indices
interventions_TOTAL.reset_index(drop=True, inplace=True)

In [ ]:
interventions_TOTAL.head(5)

In [ ]:
# Change the different notation of the levels for the variable Vector type to three consistent names: Ambulance, MUG and PIT

interventions_TOTAL.loc[interventions_TOTAL['Vector type'].isin(['AMB','Ambulance Event','Brandziekenwagen','Decontanimatieziekenwagen']),
                                                                'Vector type'] = 'Ambulance'
interventions_TOTAL.loc[interventions_TOTAL['Vector type']=='MUG Event',
                                                                'Vector type'] = 'MUG'

In [ ]:
print(interventions_TOTAL['Province intervention'].value_counts())
print(interventions_TOTAL['Vector type'].value_counts())

In [ ]:
print('Number of NaNs for Response Time: ', len(interventions_TOTAL[interventions_TOTAL['T3-T0'].isna()]))
print('Number of total known Response Times: ', len(interventions_TOTAL['T3-T0']) - len(interventions_TOTAL[interventions_TOTAL['T3-T0'].isna()]))

### 3.5 Reassign province name to location based on Belgian shapefile

### 3.5.1 Plot coordinates of all the intervention datasets

A lot of province names are wrongly assigned to the intervention location.

In [ ]:
coordinates = interventions_TOTAL[['Longitude', 'Latitude', 'Province intervention']]
coordinates.reset_index(drop=True, inplace=True) # Indices must be reset, because there were duplicates present
# Plot distribution of interventions per province
fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=coordinates, x='Longitude', y='Latitude', hue='Province intervention',ax=ax).set(xlabel='Longitude', ylabel='Latitude', title='Coordinate pairs in dataset')
belgium_with_provinces_boundary.plot(ax=ax, facecolor='none', edgecolor='black')

#### 5.3.2 Assign correct provinces to the intervention locations

Apparantly, not all province names are assigned correctly in the given dataset.
1. Create a new variable in which the correct province name is assigned.
2. In order to avoid missing values for the intervention provinces, assign to the 32 observations, wich fall just outside of Belgium, the province name of the nearest province.
3. Check visually if the correct province names have been assigned.
4. Drop the original column 'Province interventions' from the dataset.

In [ ]:
interventions_TOTAL_with_provinces = assign_province(df=interventions_TOTAL, boundaries=belgium_with_provinces_boundary)

In [ ]:
print('Missing values per variable: \n', interventions_TOTAL_with_provinces.isnull().sum())

In [ ]:
# 32 Interventions were just outside of Belgium, assign them to nearest Province
interventions_TOTAL_with_provinces = assign_nearest_province(interventions_TOTAL_with_provinces, belgium_with_provinces_boundary)
print('Missing values per variable: \n', interventions_TOTAL_with_provinces.isnull().sum())

In [ ]:
coordinates = interventions_TOTAL_with_provinces[['Longitude', 'Latitude', 'Province']]

# Plot (re)distribution of interventions per province
fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=coordinates, x='Longitude', y='Latitude', hue='Province',ax=ax).set(xlabel='Longitude', ylabel='Latitude', title='Coordinate pairs in dataset')
belgium_with_provinces_boundary.plot(ax=ax, facecolor='none', edgecolor='black')

In [ ]:
# Drop 'Province intervention column'
interventions_TOTAL_with_provinces = interventions_TOTAL_with_provinces.drop(columns='Province intervention')
interventions_TOTAL_with_provinces.head(5)

## 4. Preprocessing AED Datasets

1. Only keep the publically available AEDs in the dataset.
2. Correct format of the latitude and longitude coordinates.
3. Create new variables: Intervention, AED, Ambulance, Mug, PIT (binary indicator variables), Eventlevel, EventType, T3-T0, Vector type and Mission ID.
4. Assign the province names depending on the location of the AED to the dataset in a new column 'Province'.
5. Select only relevant variables to go further with.

In [ ]:
# Discard non-public aed locations and also those of which no data is available
yes_values = ['Y', 'y', 'Oui-Ja', 'Ja', 'Oui', 'J', np.nan]
aed_total = aed_total[
    aed_total['public'].isin(yes_values)]  

# Correcting coordinates
aed_total['Latitude'] = aed_total['latitude'].apply(correct_latitude)
aed_total['Longitude'] = aed_total['longitude'].apply(correct_longitude)

# Creatin new variables Intervention, AED, Ambulance, Mug, T3-T0, EventType and EventLevel
aed_total['Intervention'] = 0
aed_total['AED'] = 1
aed_total['Eventlevel'] = np.nan
aed_total['EventType'] = 'AED'
aed_total['T3-T0'] = pd.NaT
aed_total['Ambulance'] = 0
aed_total['Mug'] = 0
aed_total['PIT'] = 0
aed_total['Vector type'] = 'AED'
aed_total['Mission ID'] = 0

In [ ]:
# Assign provinces to AED locations
aed_total_with_provinces = assign_province(df=aed_total, boundaries=belgium_with_provinces_boundary)

In [ ]:
# Select only several variables for further analysis
aed_total_with_provinces = aed_total_with_provinces[
    ['Mission ID', 'Latitude', 'Longitude', 'Intervention', 'Eventlevel', 'T3-T0', 'EventType', 
     'Vector type', 'AED', 'Ambulance', 'Mug', 'PIT', 'Province']]

In [ ]:
aed_total_with_provinces.head(5)

## 5. Preprocessing Ambulance Dataset

1. Correct format of the latitude and longitude coordinates.
2. Create new variables: Intervention, AED, Ambulance, Mug, PIT (binary indicator variables), Eventlevel, EventType, T3-T0, Vector type and Mission ID.
4. Assign the province names depending on the location of the Ambulances to the dataset in a new column 'Province'.
5. Select only relevant variables to go further with.

In [ ]:
# Correcting coordinates
ambulance['Latitude'] = ambulance['latitude'].apply(correct_latitude)
ambulance['Longitude'] = ambulance['longitude'].apply(correct_longitude)

# Creating new variables Intervention, AED, Eventlevel, EventType, T3-T0, Ambulance and Mug
ambulance['Intervention'] = 0
ambulance['AED'] = 0
ambulance['Eventlevel'] = np.nan
ambulance['EventType'] = 'Ambulance'
ambulance['T3-T0'] = pd.NaT
ambulance['Ambulance'] = 1
ambulance['Mug'] = 0
ambulance['PIT'] = 0
ambulance['Vector type'] = 'Ambulance'
ambulance['Mission ID'] = 0

In [ ]:
# Assign provinces to Ambulance locations
ambulance_with_provinces = assign_province(df=ambulance, boundaries=belgium_with_provinces_boundary)

In [ ]:
# Select only several variables for further analysis
ambulance_with_provinces = ambulance_with_provinces[
    ['Mission ID', 'Latitude', 'Longitude', 'Intervention', 'Eventlevel', 'T3-T0', 'EventType', 
     'Vector type', 'AED', 'Ambulance', 'Mug', 'PIT', 'Province']]

In [ ]:
ambulance_with_provinces.head(5)

## 6. Preprocessing Mug Dataset

1. Correct the format of the latitude and longitude coordinates.
2. Create new variables: Intervention, AED, Ambulance, Mug, PIT (binary indicator variables), Eventlevel, EventType, T3-T0, Vector type and Mission ID.
4. Assign the province names depending on the location of the MUGs to the dataset in a new column 'Province'.
5. Select only relevant variables to go further with.

In [ ]:
# Correcting coordinates
mug1['Latitude'] = mug1['latitude'].apply(correct_latitude)
mug1['Longitude'] = mug1['longitude'].apply(correct_longitude)

# Creating new variables Intervention, AED, Eventlevel, EventType, T3-T0, Ambulance and Mug
mug1['Intervention'] = 0
mug1['AED'] = 0
mug1['Eventlevel'] = np.nan
mug1['EventType'] = 'Mug'
mug1['T3-T0'] = pd.NaT
mug1['Ambulance'] = 0
mug1['Mug'] = 1
mug1['PIT'] = 0
mug1['Vector type'] = 'MUG'
mug1['Mission ID'] = 0

In [ ]:
# Assign provinces to Ambulance locations
mug_with_provinces = assign_province(df=mug1, boundaries=belgium_with_provinces_boundary)

In [ ]:
# Select only several variables for further analysis
mug_with_provinces = mug_with_provinces[['Mission ID', 'Latitude', 'Longitude', 'Intervention', 'Eventlevel', 'T3-T0', 'EventType', 
                                         'Vector type', 'AED', 'Ambulance', 'Mug', 'PIT', 'Province']]

In [ ]:
mug_with_provinces.head(5)

## 7. Preprocessing PIT Dataset

### 7.1 Adding locations of PITs in Latitudes and Longitudes to dataframe

An external dataset was found online with the exact coordinates for the PIT locations. These latitudes and longitudes are merged to the original dataset based on the unit id of the PIT.

In [ ]:
# Extract 9-character code from 'unit_id' and 'Medical resource' from both PIT datasets and collect them in new variable
pit['id'] = pit['unit_id'].str.extract(r'(\w{9})')
pit_locations['id'] = pit_locations['Medical resource'].str.extract(r'Team: (\w{9})')

# Merge the two DataFrames based on the common 'code' column
pit_merged = pd.merge(pit, pit_locations, on='id', how='inner')

### 7.2 Other steps

1. Create new variables: Intervention, AED, Ambulance, Mug, PIT (binary indicator variables), Eventlevel, EventType, T3-T0, Vector type and Mission ID.
2. Assign the province names depending on the location of the Ambulances to the dataset in a new column 'Province'.
3. Select only several number of variables to go further with.

In [ ]:
# Creating new variables Intervention, AED, Eventlevel, EventType, T3-T0, Ambulance and Mug
pit_merged['Intervention'] = 0
pit_merged['AED'] = 0
pit_merged['Eventlevel'] = np.nan
pit_merged['EventType'] = 'PIT'
pit_merged['T3-T0'] = pd.NaT
pit_merged['Ambulance'] = 0
pit_merged['Mug'] = 0
pit_merged['PIT'] = 1
pit_merged['Vector type'] = 'PIT'
pit_merged['Mission ID'] = 0

In [ ]:
# Assign provinces to Ambulance locations
pit_merged_with_provinces = assign_province(df=pit_merged, boundaries=belgium_with_provinces_boundary)

In [ ]:
# Select only several variables for further analysis
pit_merged_with_provinces = pit_merged_with_provinces[['Mission ID', 'Latitude', 'Longitude', 'Intervention', 'Eventlevel', 'T3-T0', 
                                                       'EventType', 'Vector type', 'AED', 'Ambulance', 'Mug', 'PIT', 'Province']]
pit_merged_with_provinces.head(5)

## 8. Concatenating all datasets together for RTA and AED Placement

1. Concatenate all preprocessed datasets from before together to one big dataset containing all relevant locations.
2. Print some general information about the large dataset. 

In [ ]:
total_df = pd.concat([interventions_TOTAL_with_provinces, aed_total_with_provinces, 
                      ambulance_with_provinces, mug_with_provinces, pit_merged_with_provinces], axis=0)

In [ ]:
print('Unique number of values Latitude values: ', len(list(total_df['Latitude'].unique())))
print('\nUnique number of values Longitude values: ', len(list(total_df['Longitude'].unique())))
print('\nUnique values eventlevels', total_df['Eventlevel'].unique())
cross_tab = pd.crosstab(index=pd.Categorical(total_df["Eventlevel"]), columns='count')
print('\nCross table of Event Levels: \n', cross_tab)

max_responseTime = total_df['T3-T0'].max()
print(f'\nMaximum Response Time for total_df: {max_responseTime}')
print('\nMissing values per variable: \n', total_df.isnull().sum())
print('\nTotal Length of the Dataset: ', len(total_df))
print('\nTypes of variables in dataset: \n', total_df.dtypes)

## 9. Visual inspection of the parameters

### 9.1 Plot distribution in the form of histograms of some parameters

In [ ]:
# Setting the style for the plots
sns.set(style="whitegrid")
draw_histograms(total_df, total_df[['Latitude', 'Longitude', 'T3-T0', 'Eventlevel']], 2, 2)

In [ ]:
# Only plot Response Times for intervention locations
intervention_locations = total_df[total_df['Intervention'] == 1]

# Setting the style for the plots
sns.set(style="whitegrid")

# Create separate DataFrames for positive and negative values
positive_values = intervention_locations[intervention_locations['T3-T0'] >= 0]
negative_values = intervention_locations[intervention_locations['T3-T0'] < 0]

# Create the histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(data=positive_values, x='T3-T0', bins=20, color='green', label='Positive Values', ax=axes[0]).set(
    title='Distribution of positive Response Times', xlabel='T3-T0 (min)')
sns.histplot(data=negative_values, x='T3-T0', bins=20, color='red', label='Negative Values', ax=axes[1]).set(
    title='Distribution of negative Response Times', xlabel='T3-T0 (min)')

# Count number of negative response times
print('There are', len(negative_values), 'negative response times in the dataset')
negative_values.head(3)

In [ ]:
coordinates = total_df[['Longitude', 'Latitude']]
coordinates.reset_index(drop=True, inplace=True) # Indices must be reset, because there were duplicates present
sns.scatterplot(data=coordinates, x='Longitude', y='Latitude').set(xlabel='Longitude', ylabel='Latitude', title='Coordinate pairs in dataset')

### 9.1 Remove coordinate pairs which lie far outsdide of Belgium

Belgian coordinates found from source: https://www.belgium.be/en/about_belgium/country/geography 

Citate: 'Belgium spans 2 degrees in latitude, from 51 degrees 30 minutes N at Meerle (northernmost point) to 49 degrees 30 minutes N at Torgny (southernmost point). In longitude, it spans less than 4 degrees, from 2 degrees 33 minutes E to 6 degrees 24 minutes E.'

The nine coordinate pairs which fall far outside of Belgium (seen on plot above) are removed from dataset.

In [ ]:
# Remove locations outside of Belgium from the dataset
total_df = total_df[total_df.apply(lambda row: is_within_belgium(row['Latitude'], row['Longitude']), axis=1)]

In [ ]:
coordinates = total_df[['Longitude', 'Latitude', 'Province']]

# Plot (re)distribution of interventions per province
fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=coordinates, x='Longitude', y='Latitude', hue='Province',ax=ax).set(xlabel='Longitude', ylabel='Latitude', title='Coordinate pairs in dataset')
belgium_with_provinces_boundary.plot(ax=ax, facecolor='none', edgecolor='black')

### 9.2 Set negative Response Times to pd.NaT

1. Set the negative response times to missing variables of the type datetime.
2. Plot distribution of response times split up over the three vector types.

In [ ]:
# Transform negative time difference to NaT
total_df.loc[total_df['T3-T0'] < 0, 'T3-T0'] = pd.NaT

In [ ]:
total_df.reset_index(drop=True, inplace=True) # Reset index of dataframe

# Histogram of Response Time distribution
sns.histplot(data=total_df, x='T3-T0', bins=50, log_scale=True, kde=True, hue='Vector type').set(title='Logscale of Response Times', xlabel='Log(T3-T0)') 
# Seriously long Response times found. Seems there is a group of outliers.

## 10. Save new Dataset to the repository

In [ ]:
total_df.to_csv(output_data_path, index=False)